# 🧬 Evolutionary Design: Genotype to Phenotype
This notebook demonstrates the **Ribosome** concept in an Evolutionary Outer Loop. 

### The Core Concepts:
1. **Genotype**: The search space (a simple list of design decisions).
2. **Phenotype**: The actual Keras model built from those decisions.
3. **Evaluation**: Training the phenotype on the Jena Climate data to get a **Fitness Score**.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

print("TensorFlow version:", tf.__version__)

## 1. The Ribosome (Mapping Function)
This function translates the **Genotype** (DNA) into the **Phenotype** (Model).

In [ ]:
def build_phenotype(genotype, input_shape=(24, 7)):
    """
    Translates a Genotype list into a Keras Phenotype.
    Genotype: [model_type_idx, units, num_layers, dropout_rate, act_idx]
    """
    m_type_idx, units, n_layers, dropout, act_idx = genotype
    
    # Mapping discrete choices
    rnn_class = layers.LSTM if m_type_idx == 0 else layers.GRU
    activation = 'tanh' if act_idx == 0 else 'relu'
    
    model = models.Sequential()
    
    # Build Hidden Layers
    for i in range(n_layers):
        is_last_rnn = (i == n_layers - 1)
        model.add(rnn_class(
            units, 
            activation=activation,
            return_sequences=not is_last_rnn,
            input_shape=input_shape if i == 0 else None
        ))
        model.add(layers.Dropout(dropout))
    
    # Output Layer (Predicting Temperature)
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    
    return model

## 2. Running a Generation
We simulate an Evolutionary Algorithm selecting and building two different individuals.

In [ ]:
# Define two different 'Genotypes'
population = [
    [0, 64, 2, 0.2, 0], # Individual A: 2-layer LSTM
    [1, 128, 1, 0.1, 1] # Individual B: 1-layer GRU
]

for i, genotype in enumerate(population):
    print(f"\n--- Individual {i} ---")
    print(f"Genotype (DNA): {genotype}")
    
    # Translation to Phenotype
    phenotype = build_phenotype(genotype)
    
    # View the Phenotype structure
    phenotype.summary()
    
    # In a real run, you'd train here:
    # phenotype.fit(x_train, y_train, epochs=3)
    print(f"Phenotype {i} is ready for evaluation.")